# Data Sources and First Visual Audit

This notebook documents the first monthly dataset for the CPI local-projections project and plots the main series. The goal is a transparent data audit before estimating any impulse responses.

## Source Map

| Variable | Columns | Source | Notes |
|---|---|---|---|
| Swiss headline CPI | `cpi`, `cpi_sa` | Swiss Federal Statistical Office, LIK application data | `cpi_sa` is an STL seasonal adjustment used in the project pipeline. |
| CHF NEER | `chf_neer` | BIS effective exchange rates | Nominal effective exchange rate. Higher values mean CHF appreciation. |
| Swiss core CPI 1 | `core_cpi_1`, `core_cpi_1_sa` | SNB data portal, `plkoprex`, code `K1` | `*_sa` is an STL seasonal adjustment. |
| Swiss core CPI 2 | `core_cpi_2`, `core_cpi_2_sa` | SNB data portal, `plkoprex`, code `K2` | `*_sa` is an STL seasonal adjustment. |
| Euro area core HICP | `ea_core_hicp`, `ea_core_inflation` | ECB Data Portal, `HICP.M.U2.Y.XEF000.4F0.INX` | Inflation is computed as 12-month log change times 100. |
| Brent oil | `brent_oil`, `brent_oil_inflation` | FRED daily Brent series `DCOILBRENTEU` | Daily prices are averaged to monthly frequency; inflation is one-month log change times 100. |
| Swiss unemployment | `ch_unemployment_rate` | SNB data portal, `amarbma`, code `S1` | Useful domestic slack control. |
| Swiss energy/fuels CPI | `energy_fuel`, `energy_fuel_sa` | SNB data portal, `plkoprex`, code `ET` | Useful control or separate response channel. |

The starting merged file is `data/raw/swiss_macro_real.csv`.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update({
    "figure.figsize": (11, 4.8),
    "axes.spines.top": False,
    "axes.spines.right": False,
})

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "reports":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_PATH = PROJECT_ROOT / "data" / "raw" / "swiss_macro_real.csv"
FIGURES_DIR = PROJECT_ROOT / "outputs" / "data_overview"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

DATA_PATH

In [ ]:
data = pd.read_csv(DATA_PATH, parse_dates=["date"])
data = data.sort_values("date").reset_index(drop=True)

summary = pd.DataFrame({
    "start": data.drop(columns="date").apply(lambda s: data.loc[s.first_valid_index(), "date"] if s.first_valid_index() is not None else pd.NaT),
    "end": data.drop(columns="date").apply(lambda s: data.loc[s.last_valid_index(), "date"] if s.last_valid_index() is not None else pd.NaT),
    "missing": data.drop(columns="date").isna().sum(),
    "non_missing": data.drop(columns="date").notna().sum(),
})

print(f"Rows: {len(data):,}")
print(f"Sample: {data['date'].min().date()} to {data['date'].max().date()}")
summary.loc[
    [
        "cpi", "cpi_sa", "chf_neer", "core_cpi_1", "core_cpi_1_sa",
        "core_cpi_2", "core_cpi_2_sa", "ea_core_hicp", "ea_core_inflation",
        "brent_oil", "brent_oil_inflation", "ch_unemployment_rate",
        "energy_fuel", "energy_fuel_sa",
    ]
]

In [ ]:
def plot_pair(columns, title, ylabel="Index", filename=None):
    ax = data.set_index("date")[columns].plot(linewidth=1.8)
    ax.set_title(title)
    ax.set_xlabel("")
    ax.set_ylabel(ylabel)
    ax.legend(frameon=False)
    plt.tight_layout()
    if filename:
        plt.savefig(FIGURES_DIR / filename, dpi=160, bbox_inches="tight")
    plt.show()


def plot_single(column, title, ylabel="", filename=None):
    ax = data.set_index("date")[column].plot(linewidth=1.8)
    ax.set_title(title)
    ax.set_xlabel("")
    ax.set_ylabel(ylabel)
    plt.tight_layout()
    if filename:
        plt.savefig(FIGURES_DIR / filename, dpi=160, bbox_inches="tight")
    plt.show()

## CPI Indexes: NSA and SA

For Swiss CPI indexes, the non-seasonally adjusted and seasonally adjusted series are plotted together where both are available.

In [ ]:
plot_pair(["cpi", "cpi_sa"], "Swiss Headline CPI: NSA vs SA", filename="headline_cpi_nsa_sa.png")
plot_pair(["core_cpi_1", "core_cpi_1_sa"], "Swiss Core CPI 1: NSA vs SA", filename="core_cpi_1_nsa_sa.png")
plot_pair(["core_cpi_2", "core_cpi_2_sa"], "Swiss Core CPI 2: NSA vs SA", filename="core_cpi_2_nsa_sa.png")

## CHF Exchange Rate Measure

The baseline shock measure is the monthly log change in CHF NEER. The level is plotted first because changes are easier to interpret after checking the underlying index.

In [ ]:
data["chf_neer_change"] = 100 * (np.log(data["chf_neer"]) - np.log(data["chf_neer"]).shift(1))

In [ ]:
plot_single("chf_neer", "CHF Nominal Effective Exchange Rate", ylabel="Index", filename="chf_neer.png")
plot_single("chf_neer_change", "CHF NEER Monthly Log Change", ylabel="Percent log points", filename="chf_neer_change.png")

## External Price Controls

Euro area core inflation and Brent oil prices capture external price pressure. Brent is shown in levels and monthly inflation.

In [ ]:
plot_pair(["ea_core_hicp"], "Euro Area Core HICP", ylabel="Index", filename="ea_core_hicp.png")
plot_single("ea_core_inflation", "Euro Area Core Inflation", ylabel="12-month log change, percent", filename="ea_core_inflation.png")
plot_single("brent_oil", "Brent Oil Price", ylabel="USD per barrel", filename="brent_oil.png")
plot_single("brent_oil_inflation", "Brent Oil Monthly Inflation", ylabel="Monthly log change, percent", filename="brent_oil_inflation.png")

## Optional Baseline Controls

These are not the minimum required variables, but they are useful for the first controlled LP specification and for interpreting pass-through channels.

In [ ]:
plot_single("ch_unemployment_rate", "Swiss Unemployment Rate", ylabel="Percent", filename="ch_unemployment_rate.png")
plot_pair(["energy_fuel", "energy_fuel_sa"], "Swiss Energy and Fuels CPI: NSA vs SA", filename="energy_fuel_nsa_sa.png")

## Inflation Measures for LP Responses

The local projections will mostly use log changes of the SA CPI indexes. These quick plots check the response series that will enter the baseline and core robustness specifications.

In [ ]:
for source, target in [
    ("cpi_sa", "headline_inflation_yoy"),
    ("core_cpi_1_sa", "core_1_inflation_yoy"),
    ("core_cpi_2_sa", "core_2_inflation_yoy"),
]:
    data[target] = 100 * (np.log(data[source]) - np.log(data[source]).shift(12))

plot_pair(
    ["headline_inflation_yoy", "core_1_inflation_yoy", "core_2_inflation_yoy"],
    "Swiss CPI Inflation Measures",
    ylabel="12-month log change, percent",
    filename="swiss_cpi_inflation_yoy.png",
)